In [ ]:
# # install octave
# !sudo apt-get -qq update
# !sudo apt-get -qq install octave octave-signal liboctave-dev

# # install oct2py that compatible with colab
# import os

# from pkg_resources import get_distribution

# os.system(
#     f"pip install -qq"
#     f" ipykernel=={get_distribution('ipykernel').version}"
#     f" ipython=={get_distribution('ipython').version}"
#     f" tornado=={get_distribution('tornado').version}"
#     f" oct2py"
# )

# # install packages
# !pip install -qq matpower matpowercaseframesa

In [ ]:
import oct2py

import matpower

print(f"oct2py version: {oct2py.__version__}")
print(f"matpower version: {matpower.__version__}")

oct2py version: 5.8.0
matpower version: 8.1.0.2.2.4


In [ ]:
import os

import numpy as np
import pandas as pd
from matpowercaseframes import CaseFrames

from matpower import path_matpower, start_instance

In [ ]:
path = os.path.join(path_matpower, "data/case9.m")

In [ ]:
m = start_instance()

## Original Case

In [ ]:
cf = CaseFrames(path, load_case_engine=m)

mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()
display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,1,4,0.0000,0.0576,0.000,250,250,250,0,0,1,-360,360,71.641021,27.045924,-71.641021,-23.923127
2,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.703670,1.030006,-30.537263,-16.543365
3,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.462737,-13.456635,60.816586,-18.074836
4,3,6,0.0000,0.0586,0.000,300,300,300,0,0,1,-360,360,85.000000,-10.859709,-85.000000,14.955327
5,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.183414,3.119508,-24.095417,-24.295823
6,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.904583,-10.704177,76.379866,-0.797331
7,8,2,0.0000,0.0625,0.000,250,250,250,0,0,1,-360,360,-163.000000,9.178149,163.000000,6.653660
8,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.620134,-8.380817,-84.320163,-11.312751
9,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.679837,-38.687249,40.937352,22.893121


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,1,3,0,0,0,0,1,1.040000,0.000000,345,1,1.1,0.9
2,2,2,0,0,0,0,1,1.025000,9.280005,345,1,1.1,0.9
3,3,2,0,0,0,0,1,1.025000,4.664751,345,1,1.1,0.9
4,4,1,0,0,0,0,1,1.025788,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.012654,-3.687396,345,1,1.1,0.9
6,6,1,0,0,0,0,1,1.032353,1.966716,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.015883,0.727536,345,1,1.1,0.9
8,8,1,0,0,0,0,1,1.025769,3.719701,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995631,-3.988805,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,1,71.641021,27.045924,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,2,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,3,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


## Reduce System (Remove Transformers)

In [ ]:
# --- 1. Split branches into transformers and lines ---
is_tx = cf.branch["BR_R"] == 0
tx = cf.branch[is_tx][["F_BUS", "T_BUS"]].astype(int)
cf.branch = cf.branch[~is_tx]

# --- 2. Build bus remap: for each tx pair, src->dest
#        dest = whichever bus appears in remaining lines ---
used = set(cf.branch[["F_BUS", "T_BUS"]].values.ravel().astype(int))
remap = {
    (row.F_BUS if row.T_BUS in used else row.T_BUS): (
        row.T_BUS if row.T_BUS in used else row.F_BUS
    )
    for row in tx.itertuples()
}

# --- 3. Merge bus rows: sum load cols, propagate slack ---
sum_cols = [c for c in ["PD", "QD", "GS", "BS"] if c in cf.bus.columns]
for src, dest in remap.items():
    cf.bus.loc[dest, sum_cols] += cf.bus.loc[src, sum_cols].values
    if cf.bus.loc[src, "BUS_TYPE"] == 3:
        cf.bus.loc[dest, "BUS_TYPE"] = 3
cf.bus.drop(index=list(remap.keys()), inplace=True)

# --- 4. Remap generators, fix bus types ---
cf.gen["GEN_BUS"] = cf.gen["GEN_BUS"].astype(int).replace(remap)
has_gen = cf.bus.index.isin(cf.gen["GEN_BUS"])
cf.bus.loc[has_gen & (cf.bus["BUS_TYPE"] != 3), "BUS_TYPE"] = 2

display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
2,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.703670,1.030006,-30.537263,-16.543365
3,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.462737,-13.456635,60.816586,-18.074836
5,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.183414,3.119508,-24.095417,-24.295823
6,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.904583,-10.704177,76.379866,-0.797331
8,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.620134,-8.380817,-84.320163,-11.312751
9,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.679837,-38.687249,40.937352,22.893121


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
4,4,3,0,0,0,0,1,1.025788,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.012654,-3.687396,345,1,1.1,0.9
6,6,2,0,0,0,0,1,1.032353,1.966716,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.015883,0.727536,345,1,1.1,0.9
8,8,2,0,0,0,0,1,1.025769,3.719701,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995631,-3.988805,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,4,71.641021,27.045924,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,8,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,6,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()
display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,31.080983,9.171828,-30.879816,-24.839607
2,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.120184,-5.160393,60.499385,-26.240406
3,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.500615,-0.669696,-24.420585,-20.339586
4,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.579415,-14.660414,76.057346,3.247578
5,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.942654,-14.800422,-84.639827,-5.145166
6,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.360173,-44.854834,40.649441,28.905160


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,4,3,0,0,0,0,1,1.040000,-2.216788,345,1,1.1,0.9
2,5,1,90,30,0,0,1,1.019544,-3.599309,345,1,1.1,0.9
3,6,2,0,0,0,0,1,1.025000,2.208471,345,1,1.1,0.9
4,7,1,100,35,0,0,1,1.012276,0.912347,345,1,1.1,0.9
5,8,2,0,0,0,0,1,1.025000,3.885641,345,1,1.1,0.9
6,9,1,125,50,0,0,1,1.005122,-3.900271,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,4,71.730424,38.076988,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,8,163.000000,-11.552844,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,6,85.000000,-26.910102,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
lines_index = cf.branch.index[cf.branch["BR_R"] > 0]

cf.branch_ext = cf.branch.copy()
cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
cf.branch_ext["SFT_MAX"] = (
    cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
).pow(0.5)
cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]

,RATE_A,SFT_MAX,LOADING
branch,,,
1,250,39.787355,0.159149
2,150,65.944935,0.439633
3,150,31.843035,0.212287
4,250,77.457392,0.309830
5,250,88.193410,0.352774
6,250,60.533736,0.242135


## Derating

In [ ]:
def get_branch_thermal_correction_factor(
    t_ambient,
    t_max: float = 90.0,
    t_ref: float = 30.0,
) -> float | pd.Series:
    """
    Compute the ambient temperature correction factor (CF) for conductor ratings.

        CF = sqrt((t_max - t_ambient) / (t_max - t_ref))

    Supports scalar inputs (single CF) or per-branch inputs (Series of CFs),
    allowing each branch to have its own ambient condition and insulation class.

    For a peak hot day with heat wave scenario, this is an overestimating the line
    capability (too small derating) since it assumes:
        - ignore solar heating (overestimate)
        - assume cooling the same (overestimate, heat wave often has low wind, thus
        less cooling)
        - ignore resistive heating from current (overestimate)
        - conductor temperature is at max (underestimate, derating too much for lines
        that are not critically loaded, but fair for critically loaded and overloaded
        lines)
        - ignore radiative cooling (underestimate, radiative cooling increases with
        conductor temperature but is ignored in the formula, but this is likely a small
        effect compared to the others)
    where it overestimate is more significant than underestimate (TODO: proof this).

    Parameters
    ----------
    t_ambient : float or pd.Series
        Actual ambient temperature [°C].
        - float   : same temperature applied to all branches
        - Series  : one value per branch (must match branch DataFrame index)

    t_max : float or pd.Series, optional
        Maximum conductor temperature [°C]. Default 90.0 °C (XLPE insulation).
        - float   : same insulation class for all branches
        - Series  : per-branch insulation class

        Common insulation classes (NEC Table 310.15(B)(1)):
            - 60.0 °C : TW, UF
            - 75.0 °C : THW, THWN, XHHW (wet)
            - 90.0 °C : THWN-2, XHHW-2, USE-2  ← default

    t_ref : float or pd.Series, optional
        Reference ambient temperature of the base ratings [°C]. Default 30.0 °C.
        - NEC NFPA 70 / IEC 60364-5-52 : 30.0 °C  ← default
        - IEEE 835                     : 25.0 °C

    Returns
    -------
    float or pd.Series
        CF >= 0, dimensionless.
        - CF = 1.0 : t_ambient == t_ref     (no derating)
        - CF < 1.0 : t_ambient >  t_ref     (derating required)
        - CF > 1.0 : t_ambient <  t_ref     (uprating permitted)
        - CF = 0.0 : t_ambient >= t_max     (line must not carry current)

    Notes
    -----
    - Formula : Neher & McGrath (1957), IEEE Std 738, NEC §310.15
    - t_max   : NEC Table 310.15(B)(1)
    - t_ref   : NEC NFPA 70 / IEC 60364-5-52

    Read more
    ---------
    https://ieeexplore.ieee.org/document/4499653  # Neher & McGrath (1957)
    https://en.wikipedia.org/wiki/Neher%E2%80%93McGrath_method  # Wikipedia
    https://ecalpro.com/ko/standards/nec/table-310-15-b1-temperature  # NEC Table
    https://ieeexplore.ieee.org/document/10382442  # IEEE Std 738
    https://ieeexplore.ieee.org/document/7111195  # IEEE 835
    https://ieeexplore.ieee.org/document/11030252  # IEEE P848/D1
    """
    return np.sqrt(np.maximum(t_max - t_ambient, 0) / (t_max - t_ref))

In [ ]:
branch_temp = pd.DataFrame(
    {
        "t_ambient": [40.0, 55.0, 45.0, 50.0, 50.0, 38.0],
        "t_max": [90.0, 60.0, 75.0, 90.0, 60.0, 90.0],
        "t_ref": [30.0, 30.0, 30.0, 30.0, 30.0, 30.0],
    },
    index=lines_index,
)

In [ ]:
correction_factors = get_branch_thermal_correction_factor(
    t_ambient=branch_temp["t_ambient"],
    t_max=branch_temp["t_max"],
    t_ref=branch_temp["t_ref"],
)
correction_factors

branch
1    0.912871
2    0.408248
3    0.816497
4    0.816497
5    0.577350
6    0.930949
dtype: float64

In [ ]:
col_rates = ["RATE_A", "RATE_B", "RATE_C"]
cf.branch_ext[col_rates]

,RATE_A,RATE_B,RATE_C
branch,,,
1,250,250,250
2,150,150,150
3,150,150,150
4,250,250,250
5,250,250,250
6,250,250,250


In [ ]:
cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
cf.branch_ext.loc[lines_index, col_rates] = (
    cf.branch_ext.loc[lines_index, col_rates]
    * correction_factors[lines_index].values[:, np.newaxis]
)
cf.branch_ext[col_rates]

,RATE_A,RATE_B,RATE_C
branch,,,
1,228.217732,228.217732,228.217732
2,61.237244,61.237244,61.237244
3,122.474487,122.474487,122.474487
4,204.124145,204.124145,204.124145
5,144.337567,144.337567,144.337567
6,232.737334,232.737334,232.737334


In [ ]:
cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]

,RATE_A,SFT_MAX,LOADING
branch,,,
1,228.217732,39.787355,0.174339
2,61.237244,65.944935,1.076876
3,122.474487,31.843035,0.259997
4,204.124145,77.457392,0.379462
5,144.337567,88.193410,0.611022
6,232.737334,60.533736,0.260095


## Cascading

In [ ]:
# overloaded lines status update
overloaded = cf.branch_ext.loc[:, "LOADING"] > 1.0
cf.branch.loc[overloaded, "BR_STATUS"] = 0

In [ ]:
mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()

cf.branch_ext = cf.branch.copy()
cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
cf.branch_ext["SFT_MAX"] = (
    cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
).pow(0.5)

cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
cf.branch_ext.loc[lines_index, col_rates] = (
    cf.branch_ext.loc[lines_index, col_rates]
    * correction_factors[lines_index].values[:, np.newaxis]
)

cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]

,RATE_A,SFT_MAX,LOADING
branch,,,
1,228.217732,96.250393,0.421748
2,61.237244,0.000000,0.000000
3,122.474487,85.663630,0.699441
4,204.124145,29.070393,0.142415
5,144.337567,147.609194,1.022666
6,232.737334,63.776253,0.274027


In [ ]:
# overloaded lines status update
overloaded = cf.branch_ext.loc[:, "LOADING"] > 1.0
cf.branch.loc[overloaded, "BR_STATUS"] = 0

In [ ]:
cf.branch[["F_BUS", "T_BUS", "BR_STATUS"]]

,F_BUS,T_BUS,BR_STATUS
branch,,,
1,4,5,1
2,5,6,0
3,6,7,1
4,7,8,1
5,8,9,0
6,9,4,1


In [ ]:
[groups, isolated] = m.find_islands(cf.to_mpc(), nout=2)
groups, isolated

(Cell([[array([[1., 2., 6.]]), array([[3., 4., 5.]])]]),
 array([], shape=(1, 0), dtype=float64))

In [ ]:
mpcs = m.extract_islands(mpc, groups)

cfs = {}
for isl, mpc in enumerate(mpcs.flatten()):
    cfs[isl] = CaseFrames(mpc)
    cfs[isl].infer_numpy()

    cfs[isl].bus.set_index(cfs[isl].bus["BUS_I"].astype(int), drop=False, inplace=True)
    cfs[isl].bus.index.name = "bus"

    display(cfs[isl].branch[["F_BUS", "T_BUS", "BR_STATUS"]])
    display(cfs[isl].bus)
    display(cfs[isl].gen)

,F_BUS,T_BUS,BR_STATUS
branch,,,
1,4,5,1
2,9,4,1


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
4,4,3,0,0,0,0,1,1.040000,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.001440,-6.569568,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.996056,-1.192909,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,4,76.237479,67.64341,300,-300,1.04,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0


,F_BUS,T_BUS,BR_STATUS
branch,,,
1,6,7,1
2,7,8,1


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
6,6,2,0,0,0,0,1,1.025000,16.251493,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.011716,11.557860,345,1,1.1,0.9
8,8,2,0,0,0,0,1,1.025000,12.108770,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,8,163,-0.600844,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
2,6,85,-4.055204,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
cf = cfs[0]
if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
    slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
    slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
    cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus
display(cf.bus)

,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
4,4,3,0,0,0,0,1,1.040000,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.001440,-6.569568,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.996056,-1.192909,345,1,1.1,0.9


In [ ]:
cf = cfs[1]
if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
    slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
    slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
    cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus
display(cf.bus)

,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
6,6,2,0,0,0,0,1,1.025000,16.251493,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.011716,11.557860,345,1,1.1,0.9
8,8,3,0,0,0,0,1,1.025000,12.108770,345,1,1.1,0.9


In [ ]:
for _isl, cf in cfs.items():
    if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
        slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
        slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
        cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus

    m.runpf(cf.to_mpc(), verbose=False)
    mpc = m.runpf(cf.to_mpc(), verbose=False)
    cf = CaseFrames(mpc)
    cf.infer_numpy()

    cf.branch_ext = cf.branch.copy()
    cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
    cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
    cf.branch_ext["SFT_MAX"] = (
        cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
    ).pow(0.5)

    lines_index = cf.branch.index[cf.branch["BR_R"] > 0]

    cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
    cf.branch_ext.loc[lines_index, col_rates] = (
        cf.branch_ext.loc[lines_index, col_rates]
        * correction_factors[lines_index].values[:, np.newaxis]
    )

    cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
    display(
        cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]
    )  # see, we can further cascade

,RATE_A,SFT_MAX,LOADING
branch,,,
1,228.217732,96.250393,0.421748
2,102.062073,136.285396,1.335319


,RATE_A,SFT_MAX,LOADING
branch,,,
1,136.930639,85.663630,0.625599
2,102.062073,29.070393,0.284831
